# Demo 03 : Checkpointers

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · threads

We keep ARI510 demo22's checkpointer structure and give a service assistant separate threads. The same thread retains messages across calls.

In [2]:
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.checkpoint.memory import InMemorySaver

def answer(state: MessagesState):
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("answer", answer)
builder.add_node("record", lambda state: {})
builder.add_edge(START, "answer")
builder.add_edge("answer", "record")
builder.add_edge("record", END);

In [3]:
graph = builder.compile(checkpointer=InMemorySaver())
morning = {"configurable": {"thread_id": "u1-morning"}}
afternoon = {"configurable": {"thread_id": "u1-afternoon"}}
graph.invoke({"messages": [("user", "My bus is 4012 and my depot is Noord. Acknowledge briefly.")]}, morning)
remembered = graph.invoke({"messages": [("user", "Which bus and depot did I mention?")]}, morning)
separate = graph.invoke({"messages": [("user", "Which bus and depot did I mention? If unknown, say so.")]}, afternoon)
print("same thread :", remembered["messages"][-1].content)
print("new thread :", separate["messages"][-1].content)
snapshot = graph.get_state(morning)
print("saved messages :", len(snapshot.values["messages"]))
print("next :", snapshot.next)
print("checkpoint :", snapshot.config["configurable"]["checkpoint_id"])

same thread : Bus 4012, depot Noord.
new thread : You didn’t mention a bus or depot.
saved messages : 4
next : ()
checkpoint : 1f1d1882-3824-6d91-8006-6e57e8f862e5


We compare the saved messages as well as the replies. A model can guess a bus or depot on a new thread, but the checkpoint shows which context it received.

## b · sqlite-saver

We close a SQLite connection after a saved pause. We open the same file and compile a second graph to restore and finish the run.

In [4]:
from pathlib import Path
from uuid import uuid4
from langgraph.checkpoint.sqlite import SqliteSaver

notebook_dir = ROOT / "notebooks" / "03-state-management" if ROOT else Path("notebooks/03-state-management")
notebook_dir.mkdir(parents=True, exist_ok=True)
database = notebook_dir / "module03.db"
config = {"configurable": {"thread_id": f"sqlite-{uuid4().hex}"}}
with SqliteSaver.from_conn_string(str(database)) as saver:
    persistent = builder.compile(checkpointer=saver, interrupt_after=["answer"])
    persistent.invoke({"messages": [("user", "Write one short acknowledgement of brake inspection for bus 4012.")]}, config)
    print("saved next :", persistent.get_state(config).next)

with SqliteSaver.from_conn_string(str(database)) as saver:
    restored = builder.compile(checkpointer=saver)
    snapshot = restored.get_state(config)
    print("restored next :", snapshot.next)
    print("saved reply :", snapshot.values["messages"][-1].content)
    restored.invoke(None, config)
    print("finished next :", restored.get_state(config).next)
print("file exists :", database.exists())

saved next : ('record',)
restored next : ('record',)
saved reply : Brake inspection acknowledged for bus 4012.
finished next : ()
file exists : True


## Things to try

Optional : we can change a ticket, a thread id or a returned update and compare the saved state.